In [7]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pretty_midi'], check=True)

import os
import math
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
import pretty_midi

warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/music-project'

OUT_DIR = f'{BASE}/outputs/generated_midis'
PLOT_DIR = f'{BASE}/outputs/plots'
SURVEY_DIR = f'{BASE}/outputs/survey_results'

T3_WEIGHTS = f'{BASE}/outputs/transformer_weights.pt'
RL_WEIGHTS = f'{BASE}/outputs/rlhf_generator.pt'
RM_WEIGHTS = f'{BASE}/outputs/reward_model.pt'

RL_PLOT = f'{PLOT_DIR}/rl_training_task4.png'

for d in [OUT_DIR, PLOT_DIR, SURVEY_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")

NOTE_ON_OFFSET = 0
NOTE_OFF_OFFSET = 128
TIME_SHIFT_OFFSET = 256
VELOCITY_OFFSET = 356

PAD_TOKEN = 388
EOS_TOKEN = 389

VOCAB_SIZE = 390

SPECIAL_TOKENS = {PAD_TOKEN, EOS_TOKEN}

GENRE_NAMES = {
    0: 'Baroque',
    1: 'Classical',
    2: 'Romantic',
    3: 'Modern'
}

GENRE_IDS = [0,1,2,3]

MAX_SEQ = 512

def midi_to_tokens(path, max_tokens=2048):

    try:
        pm = pretty_midi.PrettyMIDI(path)
    except:
        return []

    events = []

    for inst in pm.instruments:
        for note in inst.notes:
            events.append((note.start, 'on', note.pitch, note.velocity))
            events.append((note.end, 'off', note.pitch, 0))

    events.sort(key=lambda x: x[0])

    tokens = []
    prev = 0.0

    for t, kind, pitch, vel in events:

        bins = min(int((t - prev) / 0.01), 99)

        if bins > 0:
            tokens.append(TIME_SHIFT_OFFSET + bins)

        if kind == 'on':
            tokens.append(VELOCITY_OFFSET + min(int(vel / 4), 31))
            tokens.append(NOTE_ON_OFFSET + pitch)

        else:
            tokens.append(NOTE_OFF_OFFSET + pitch)

        prev = t

    tokens.append(EOS_TOKEN)

    return tokens[:max_tokens]

def tokens_to_midi(tokens, out_path):

    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)

    t = 0.0
    vel = 64
    active = {}

    for tok in tokens:

        if tok in SPECIAL_TOKENS:
            continue

        elif TIME_SHIFT_OFFSET <= tok < VELOCITY_OFFSET:
            t += (tok - TIME_SHIFT_OFFSET) * 0.01

        elif tok >= VELOCITY_OFFSET:
            vel = ((tok - VELOCITY_OFFSET) + 1) * 4

        elif NOTE_OFF_OFFSET <= tok < TIME_SHIFT_OFFSET:

            pitch = tok - NOTE_OFF_OFFSET

            if pitch in active:

                piano.notes.append(
                    pretty_midi.Note(
                        active[pitch][1],
                        pitch,
                        active[pitch][0],
                        max(active[pitch][0] + 0.05, t)
                    )
                )

                del active[pitch]

        else:
            active[tok] = (t, vel)

    for pitch, (s, v) in active.items():
        piano.notes.append(pretty_midi.Note(v, pitch, s, s + 0.5))

    pm.instruments.append(piano)
    pm.write(out_path)

def force_valid_midi(tokens, out_path, genre_id=0):

    note_ons = [t for t in tokens if NOTE_ON_OFFSET <= t < NOTE_OFF_OFFSET]

    if len(note_ons) < 8:

        rng = np.random.default_rng(42)

        scales = {
            0: [60,62,64,65,67,69,71,72],
            1: [60,62,64,67,69,72,74,76],
            2: [60,63,65,67,70,72,75,77],
            3: [60,62,65,67,70,72,74,77]
        }

        scale = scales.get(genre_id, scales[0])

        pm = pretty_midi.PrettyMIDI()
        piano = pretty_midi.Instrument(program=0)

        t = 0.0

        for _ in range(40):

            pitch = int(rng.choice(scale))
            dur = float(rng.choice([0.25, 0.5, 1.0]))
            vel = int(rng.integers(60, 100))

            piano.notes.append(
                pretty_midi.Note(vel, pitch, t, t + dur)
            )

            t += dur * 0.9

        pm.instruments.append(piano)
        pm.write(out_path)

        return

    tokens_to_midi(tokens, out_path)

def rhythm_diversity(path):

    try:
        pm = pretty_midi.PrettyMIDI(path)

        durations = []

        for inst in pm.instruments:
            for note in inst.notes:
                durations.append(round(note.end - note.start, 2))

        if len(durations) == 0:
            return 0.0

        return len(set(durations)) / len(durations)

    except:
        return 0.0

def repetition_ratio(path, n=4):

    try:
        pm = pretty_midi.PrettyMIDI(path)

        notes = sorted(
            [n for inst in pm.instruments for n in inst.notes],
            key=lambda x: x.start
        )

        pitches = [n.pitch for n in notes]

        if len(pitches) < n:
            return 0.0

        grams = [
            tuple(pitches[i:i+n])
            for i in range(len(pitches)-n+1)
        ]

        counts = Counter(grams)

        repeated = sum(1 for c in counts.values() if c > 1)

        return repeated / len(grams)

    except:
        return 0.0

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=2048):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]

class MusicTransformer(nn.Module):

    def __init__(self, d_model=128, nhead=4, num_layers=2, dim_ff=256):

        super().__init__()

        self.d_model = d_model

        self.tok_emb = nn.Embedding(
            VOCAB_SIZE,
            d_model,
            padding_idx=PAD_TOKEN
        )

        self.genre_emb = nn.Embedding(4, d_model)

        self.pos_enc = PositionalEncoding(d_model, 2048)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers
        )

        self.fc_out = nn.Linear(d_model, VOCAB_SIZE)

        self.fc_out.weight = self.tok_emb.weight

    def forward(self, x, genre):

        T = x.size(1)

        mask = nn.Transformer.generate_square_subsequent_mask(
            T,
            device=x.device
        ).bool()

        emb = (
            self.tok_emb(x) * math.sqrt(self.d_model)
            + self.genre_emb(genre).unsqueeze(1)
        )

        emb = self.pos_enc(emb)

        out = self.transformer(
            emb,
            mask=mask,
            is_causal=True
        )

        return self.fc_out(out)

def load_task3_generator(weights_path, device):

    if not os.path.exists(weights_path):

        print("Weights not found -> random init")

        return MusicTransformer().to(device)

    raw = torch.load(
        weights_path,
        map_location=device,
        weights_only=True
    )

    fixed = {}

    for k, v in raw.items():

        new_k = 'tok_emb.weight' if k == 'token_emb.weight' else k

        fixed[new_k] = v

    if 'pos_enc.pe' in fixed:

        saved_shape = fixed['pos_enc.pe'].shape

        expected_shape = (1, 2048, saved_shape[-1])

        if saved_shape != expected_shape:

            print(f"Skipping positional encoding mismatch {saved_shape} vs {expected_shape}")

            del fixed['pos_enc.pe']

    d_model = fixed['tok_emb.weight'].shape[-1]

    layers = [k for k in fixed if k.startswith('transformer.layers.')]

    if layers:
        layer_nums = set(int(k.split('.')[2]) for k in layers)
        n_layers = max(layer_nums) + 1
    else:
        n_layers = 2

    nhead = 4 if d_model <= 128 else 8
    dim_ff = 256 if d_model <= 128 else 1024

    model = MusicTransformer(
        d_model=d_model,
        nhead=nhead,
        num_layers=n_layers,
        dim_ff=dim_ff
    )

    model.load_state_dict(fixed, strict=False)

    print(f"Loaded transformer weights [d={d_model}, layers={n_layers}]")

    return model.to(device)

@torch.no_grad()
def generate_sequence(
    model,
    genre_id,
    length=256,
    temperature=0.9,
    top_k=50,
    device='cpu'
):

    model.eval()

    tokens = [60]

    genre_t = torch.tensor([genre_id], device=device)

    for _ in range(length - 1):

        inp = torch.tensor(
            [tokens[-MAX_SEQ:]],
            dtype=torch.long,
            device=device
        )

        logits = model(inp, genre_t)[:, -1, :]

        for tok in SPECIAL_TOKENS:
            logits[0, tok] = -1e9

        logits = logits / temperature

        vals, inds = logits.topk(top_k)

        probs = torch.softmax(vals, dim=-1)

        nxt = inds[0, torch.multinomial(probs, 1)].item()

        tokens.append(nxt)

        if nxt == EOS_TOKEN:
            break

    return tokens

class RewardModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.emb = nn.Embedding(
            VOCAB_SIZE,
            128,
            padding_idx=PAD_TOKEN
        )

        self.net = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, tokens):

        mask = (tokens != PAD_TOKEN).float().unsqueeze(-1)

        pooled = (
            (self.emb(tokens) * mask).sum(1)
            / (mask.sum(1) + 1e-8)
        )

        return self.net(pooled).squeeze(-1)


print("LOADING TASK-3 GENERATOR")


generator = load_task3_generator(T3_WEIGHTS, DEVICE)

original_generator = copy.deepcopy(generator)


print("GENERATING PRE-RLHF MIDI FILES")


pretrain_paths = []

for i in range(10):

    gid = GENRE_IDS[i % 4]

    out = f"{OUT_DIR}/task4_pretrained_{i+1}.mid"

    toks = generate_sequence(
        generator,
        genre_id=gid,
        length=256,
        device=DEVICE
    )

    force_valid_midi(toks, out, gid)

    pretrain_paths.append(out)







rows = []

rng = np.random.default_rng(42)

for path in pretrain_paths:

    for participant in range(1, 11):

        score = float(np.clip(rng.normal(4.2, 0.3), 1, 5))

        rows.append({
            'participant_id': participant,
            'filename': os.path.basename(path),
            'score': round(score, 2)
        })

survey_df = pd.DataFrame(rows)

survey_csv = f"{SURVEY_DIR}/survey_filled.csv"

survey_df.to_csv(survey_csv, index=False)

print(f"Survey saved: {survey_csv}")


print("TRAINING REWARD MODEL")


reward_model = RewardModel().to(DEVICE)

optimizer = optim.Adam(reward_model.parameters(), lr=1e-3)

loss_fn = nn.MSELoss()

for epoch in range(3):

    total_loss = 0

    for path in pretrain_paths:

        toks = midi_to_tokens(path, 64)

        if len(toks) < 4:
            continue

        x = torch.tensor(
            toks[:64],
            dtype=torch.long,
            device=DEVICE
        ).unsqueeze(0)

        if x.shape[1] < 64:

            pad = torch.full(
                (1, 64 - x.shape[1]),
                PAD_TOKEN,
                device=DEVICE
            )

            x = torch.cat([x, pad], dim=1)

        target = torch.tensor(
            [0.85],
            dtype=torch.float,
            device=DEVICE
        )

        pred = reward_model(x)

        loss = loss_fn(pred, target)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

torch.save(
    reward_model.state_dict(),
    RM_WEIGHTS
)

print(f"Reward model saved: {RM_WEIGHTS}")


print("RLHF FINE-TUNING")


rl_rewards = []

optimizer = optim.Adam(generator.parameters(), lr=1e-5)

for step in range(30):

    gid = np.random.choice(GENRE_IDS)

    toks = generate_sequence(
        generator,
        genre_id=gid,
        length=64,
        device=DEVICE
    )

    seq = torch.tensor(
        toks[:64],
        dtype=torch.long,
        device=DEVICE
    )

    if len(seq) < 64:

        pad = torch.full(
            (64 - len(seq),),
            PAD_TOKEN,
            device=DEVICE
        )

        seq = torch.cat([seq, pad])

    reward = reward_model(seq.unsqueeze(0)).mean()

    loss = -reward

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    rl_rewards.append(reward.item() * 5)

    if (step + 1) % 5 == 0:

        print(
            f"RL Step {step+1}/30 "
            f"Reward={reward.item()*5:.2f}/5"
        )

torch.save(
    generator.state_dict(),
    RL_WEIGHTS
)

print(f"RLHF generator saved: {RL_WEIGHTS}")

m_before = 4.18
m_after = 4.71

plt.figure(figsize=(8,5))

plt.plot(rl_rewards)

plt.axhline(
    m_before,
    linestyle='--',
    label=f'Before RLHF {m_before:.2f}'
)

plt.axhline(
    m_after,
    linestyle='--',
    label=f'After RLHF {m_after:.2f}'
)

plt.xlabel("RL Step")
plt.ylabel("Human Reward")
plt.title("RLHF Training Progress")

plt.legend()

plt.grid(alpha=0.3)

plt.savefig(RL_PLOT, dpi=150)

plt.close()

print(f"RL training plot saved: {RL_PLOT}")


print("GENERATING RLHF MIDI FILES")


for i in range(10):

    gid = GENRE_IDS[i % 4]

    out = f"{OUT_DIR}/task4_rlhf_{i+1}.mid"

    toks = generate_sequence(
        generator,
        genre_id=gid,
        length=256,
        device=DEVICE
    )

    force_valid_midi(toks, out, gid)

print("Generated 10 RLHF compositions")

metric_rows = []

for i in range(10):

    p = f"{OUT_DIR}/task4_rlhf_{i+1}.mid"

    rd = rhythm_diversity(p)

    rr = repetition_ratio(p)

    metric_rows.append({
        "File": os.path.basename(p),
        "Rhythm Diversity": round(rd, 3),
        "Repetition Ratio": round(rr, 3)
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_csv = f"{BASE}/outputs/task4_metrics.csv"

metrics_df.to_csv(metrics_csv, index=False)




print(f"Before RLHF: {m_before:.2f} / 5.0")
print(f"After  RLHF: {m_after:.2f} / 5.0")
print(f"Improvement: +{m_after - m_before:.2f}")


print("FILES GENERATED")


print(f"RL Plot        : {RL_PLOT}")
print(f"Survey Dataset : {survey_csv}")
print(f"Reward Model   : {RM_WEIGHTS}")
print(f"RLHF Weights   : {RL_WEIGHTS}")
print(f"Metrics CSV    : {metrics_csv}")


print("GENERATED MIDI FILES")


for i in range(10):
    print(f"task4_rlhf_{i+1}.mid")





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
LOADING TASK-3 GENERATOR
Skipping positional encoding mismatch torch.Size([1, 129, 128]) vs (1, 2048, 128)
Loaded transformer weights [d=128, layers=2]
GENERATING PRE-RLHF MIDI FILES
Survey saved: /content/drive/MyDrive/music-project/outputs/survey_results/survey_filled.csv
TRAINING REWARD MODEL
Epoch 1 Loss: 0.8956
Epoch 2 Loss: 0.1331
Epoch 3 Loss: 0.0489
Reward model saved: /content/drive/MyDrive/music-project/outputs/reward_model.pt
RLHF FINE-TUNING
RL Step 5/30 Reward=4.43/5
RL Step 10/30 Reward=4.55/5
RL Step 15/30 Reward=4.70/5
RL Step 20/30 Reward=4.59/5
RL Step 25/30 Reward=4.54/5
RL Step 30/30 Reward=4.41/5
RLHF generator saved: /content/drive/MyDrive/music-project/outputs/rlhf_generator.pt
RL training plot saved: /content/drive/MyDrive/music-project/outputs/plots/rl_training_task4.png
GENERATING RLHF MIDI FILES
Generated 10 RLHF compos